# Visual AU Extraction — IEMOCAP (OpenFace 2.0 fallback)

Fallback AU pipeline using OpenFace 2.0 CLI binary.
Run this instead of Section 3 in `extract_video_iemocap.ipynb` when OpenFace 3.0 is unavailable.

**Prerequisite:** Compile OpenFace 2.0 and set `OPENFACE2_BIN` in the config cell.
```bash
sudo apt-get install cmake libopenblas-dev libopencv-dev libdlib-dev
git clone https://github.com/TadasBaltrusaitis/OpenFace.git
cd OpenFace && mkdir build && cd build
cmake -D CMAKE_BUILD_TYPE=Release .. && make -j$(nproc)
# Binary at: OpenFace/build/bin/FeatureExtraction
```

IEMOCAP videos are speaker-cropped — expect >95% face coverage.

Output: `Dataset/Processed/IEMOCAP/features/video_openface2_au.pt` — `{utt_id: np.array(18,)}`

In [1]:
import os, sys, subprocess, tempfile, torch, numpy as np, pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

PROJECT_ROOT = Path('/mnt/Work/ML/Thesis/BMVC/Hopeful')
DATA_ROOT    = PROJECT_ROOT / 'Dataset/Processed/IEMOCAP'
FEATURES_DIR = DATA_ROOT / 'features'
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

OPENFACE2_BIN     = Path('/path/to/OpenFace/build/bin/FeatureExtraction')  # EDIT THIS
AU_CONF_THRESHOLD = 0.9

print(f'OpenFace2 bin: {OPENFACE2_BIN}')
print(f'Exists:        {OPENFACE2_BIN.exists()}')

labels = pd.read_csv(DATA_ROOT / 'labels.csv')
print(f'Total utterances: {len(labels)}')

OpenFace2 bin: /path/to/OpenFace/build/bin/FeatureExtraction
Exists:        False
Total utterances: 10039


In [2]:
AU_COLS = [
    'AU01_r', 'AU02_r', 'AU04_r', 'AU05_r', 'AU06_r', 'AU07_r',
    'AU09_r', 'AU10_r', 'AU12_r', 'AU14_r', 'AU15_r', 'AU17_r',
    'AU20_r', 'AU23_r', 'AU25_r', 'AU26_r', 'AU28_r', 'AU45_r',
]
AU_DIM = len(AU_COLS)  # 18
print(f'AU dim: {AU_DIM}')

def extract_au_of2(video_path: Path):
    with tempfile.TemporaryDirectory() as tmpdir:
        subprocess.run(
            [str(OPENFACE2_BIN), '-f', str(video_path), '-out_dir', tmpdir, '-aus'],
            capture_output=True, timeout=300,
        )
        csv_files = list(Path(tmpdir).glob('*.csv'))
        if not csv_files:
            return None
        df = pd.read_csv(csv_files[0])
        df.columns = df.columns.str.strip()
        if 'confidence' in df.columns:
            df = df[df['confidence'] >= AU_CONF_THRESHOLD]
        if 'success' in df.columns:
            df = df[df['success'] == 1]
        if len(df) == 0:
            return None
        vec = np.zeros(AU_DIM, dtype=np.float32)
        for i, col in enumerate(AU_COLS):
            if col in df.columns:
                vec[i] = float(df[col].mean())
        return vec

AU dim: 18


In [3]:
assert OPENFACE2_BIN.exists(), f'Binary not found: {OPENFACE2_BIN}'

out_path = FEATURES_DIR / 'video_openface2_au.pt'
if out_path.exists():
    print('Already exists — skipping')
else:
    features  = {}
    skipped   = []
    zero_face = []

    for _, row in tqdm(labels.iterrows(), total=len(labels), desc='AU OF2'):
        vid_path = DATA_ROOT / row['video_path']
        if not vid_path.exists():
            skipped.append(row['utt_id']); continue
        au_vec = extract_au_of2(vid_path)
        if au_vec is None:
            zero_face.append(row['utt_id'])
            features[row['utt_id']] = np.zeros(AU_DIM, dtype=np.float32)
            continue
        features[row['utt_id']] = au_vec

    torch.save(features, out_path)
    cov = len(features) - len(zero_face)
    print(f'Saved: {out_path}  ({out_path.stat().st_size/1e6:.1f} MB, {len(features)} entries)')
    print(f'Face coverage: {cov}/{len(features)}  ({cov/max(len(features),1)*100:.1f}%)')
    if skipped:   print(f'Skipped (missing): {skipped}')
    if zero_face: print(f'No face detected:  {len(zero_face)} utterances')

AssertionError: Binary not found: /path/to/OpenFace/build/bin/FeatureExtraction

## Verification

In [ ]:
print('=== Verification ===')
pt = FEATURES_DIR / 'video_openface2_au.pt'
if not pt.exists():
    print('  MISSING')
else:
    d = torch.load(pt, weights_only=False)
    all_feats = np.stack(list(d.values()))
    print(f'  count={len(d)}/10039  shape={next(iter(d.values())).shape}  '
          f'nan={np.isnan(all_feats).any()}  inf={np.isinf(all_feats).any()}')